# **Hyperparameter tuning in sklearn**

In Machine Learning, a **hyperparameter** is a parameter of a ML model whose value is used to customize the learning algorithm.

Typically, every application domain, or even every single problem, has a optimum set of hyperparameters values that provide the best performance for the selected model on that problem or dataset.

The best hyperparameters values can not be extracted from the dataset characteristics, all that you have is your experience and some heuristics. Therefore, **automation toolkits that boost the hyperparameters search** are a key resource for the ML engineer. `GridSearchCV` and `RandomizedSearchCV` are the `sklearn` packages for that purpose.





## Import Library







In [ ]:
!pip install --upgrade openpyxl

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


## Import data from the dataset






In [ ]:
# upload files from your local machine
#from google.colab import files
#files.upload()
dataset = pd.read_excel('Ensayo_Motores_rpm_par.xlsx')
X = dataset.iloc[:, [1,2]].values
y = dataset.iloc[:, 3].values


In [ ]:
# explore dataset
np.set_printoptions(formatter={'float': '{:0.2f}'.format})
print(X)
print(y)
dataset.info()
dataset.head(10)

[[19.00 19000.00]
 [35.00 20000.00]
 [26.00 43000.00]
 [27.00 57000.00]
 [19.00 76000.00]
 [27.00 58000.00]
 [27.00 84000.00]
 [32.00 150000.00]
 [25.00 33000.00]
 [35.00 65000.00]
 [26.00 80000.00]
 [26.00 52000.00]
 [20.00 86000.00]
 [32.00 18000.00]
 [18.00 82000.00]
 [29.00 80000.00]
 [47.00 25000.00]
 [45.00 26000.00]
 [46.00 28000.00]
 [48.00 29000.00]
 [45.00 22000.00]
 [47.00 49000.00]
 [48.00 41000.00]
 [45.00 22000.00]
 [46.00 23000.00]
 [47.00 20000.00]
 [49.00 28000.00]
 [47.00 30000.00]
 [29.00 43000.00]
 [31.00 18000.00]
 [31.00 74000.00]
 [27.00 137000.00]
 [21.00 16000.00]
 [28.00 44000.00]
 [27.00 90000.00]
 [35.00 27000.00]
 [33.00 28000.00]
 [30.00 49000.00]
 [26.00 72000.00]
 [27.00 31000.00]
 [27.00 17000.00]
 [33.00 51000.00]
 [35.00 108000.00]
 [30.00 15000.00]
 [28.00 84000.00]
 [23.00 20000.00]
 [25.00 79000.00]
 [27.00 54000.00]
 [30.00 135000.00]
 [31.00 89000.00]
 [24.00 32000.00]
 [18.00 44000.00]
 [29.00 83000.00]
 [35.00 23000.00]
 [27.00 58000.00]
 [24.0

,Nº Serie Motor,Par Carga (N.m),RPM (Revoluciones/min),Fallo Motor
0,15624510,19.0,19000.0,0
1,15810944,35.0,20000.0,0
2,15668575,26.0,43000.0,0
3,15603246,27.0,57000.0,0
4,15804002,19.0,76000.0,0
5,15728773,27.0,58000.0,0
6,15598044,27.0,84000.0,0
7,15694829,32.0,150000.0,1
8,15600575,25.0,33000.0,0
9,15727311,35.0,65000.0,0


## Data preprocessing and Data cleaning (Not required)
#### - No missing data (NaN).
#### - No categorical data.
#### - No dummy data.







## Split data into training/testing sets

In [ ]:
#Split data into training/testing
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 0)

## Variable scaling

In [ ]:
from sklearn.preprocessing import StandardScaler
sc_X = StandardScaler()
X_train_sc = sc_X.fit_transform(X_train)
X_test_sc = sc_X.transform(X_test)
np.set_printoptions(formatter=None)
print(X_train_sc)
print(X_test_sc)

[[ 0.58164944 -0.88670699]
 [-0.60673761  1.46173768]
 [-0.01254409 -0.5677824 ]
 [-0.60673761  1.89663484]
 [ 1.37390747 -1.40858358]
 [ 1.47293972  0.99784738]
 [ 0.08648817 -0.79972756]
 [-0.01254409 -0.24885782]
 [-0.21060859 -0.5677824 ]
 [-0.21060859 -0.19087153]
 [-0.30964085 -1.29261101]
 [-0.30964085 -0.5677824 ]
 [ 0.38358493  0.09905991]
 [ 0.8787462  -0.59677555]
 [ 2.06713324 -1.17663843]
 [ 1.07681071 -0.13288524]
 [ 0.68068169  1.78066227]
 [-0.70576986  0.56295021]
 [ 0.77971394  0.35999821]
 [ 0.8787462  -0.53878926]
 [-1.20093113 -1.58254245]
 [ 2.1661655   0.93986109]
 [-0.01254409  1.22979253]
 [ 0.18552042  1.08482681]
 [ 0.38358493 -0.48080297]
 [-0.30964085 -0.30684411]
 [ 0.97777845 -0.8287207 ]
 [ 0.97777845  1.8676417 ]
 [-0.01254409  1.25878567]
 [-0.90383437  2.27354572]
 [-1.20093113 -1.58254245]
 [ 2.1661655  -0.79972756]
 [-1.39899564 -1.46656987]
 [ 0.38358493  2.30253886]
 [ 0.77971394  0.76590222]
 [-1.00286662 -0.30684411]
 [ 0.08648817  0.76590222]
 

### **First attempt: Create and fit a base SVM model**

In [ ]:
from sklearn import svm

model_svc = svm.SVC(kernel = "linear", probability=True, random_state = 0)
model_svc.fit(X_train_sc,y_train)
model_svc.score(X_test_sc, y_test)

0.9

In [ ]:
model_svc = svm.SVC(kernel = "rbf", probability=True, random_state = 0)
model_svc.fit(X_train_sc,y_train)
model_svc.score(X_test_sc, y_test)

0.93

### **Second attempt: Create and fit a basic Random Forest model**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(n_estimators = 10, criterion = "entropy", random_state = 0)
model_rf.fit(X_train_sc, y_train)
model_rf.score(X_test_sc, y_test)

0.91

Not bad, pretty good results. But, how do we know if there is a better model?

and how do we tune the model's hyperparameters to get best accuracy?

### **Solution: Perform model and hyperparameters exhaustive search**

We need to perform an exhaustive model and parameters search to ensure best accuracy.

So, first, let's define the subset of models and parameters for each model where we will perform our search.






In [ ]:
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# define model dictionary
# liblinear: es un optimizador para problemas de clasificacion binaria. No funciona con multiclase
# gamma
#  - Gamma alto: Cada punto influye solo en su entorno inmediato, generando fronteras de decisión (plano de corte) muy complejas y riesgo de sobreajuste.
#  - Gamma bajo: La influencia de cada punto se extiende más, produciendo fronteras de decisión suaves y mayor capacidad de generalización.
models = {
    'LogisticRegression': LogisticRegression(solver='liblinear',multi_class='auto', random_state = 0 ),
    'RandomForestClassifier': RandomForestClassifier(criterion = "entropy", random_state = 0 ),
    'SVM' : svm.SVC(gamma='auto', random_state = 0 )
}
# define parameters dictionary
params = {
    'LogisticRegression': { 'C': [1, 5, 10] },
    'RandomForestClassifier': { 'n_estimators': [1, 5, 10] },
    'SVM' : { 'C': [1,10,20], 'kernel': ['rbf','linear'] }
}

Just for sanity, let's ensure that the dictionaries are well defined:

In [ ]:
models.keys()

dict_keys(['LogisticRegression', 'RandomForestClassifier', 'SVM'])

In [ ]:
models.values()

dict_values([LogisticRegression(multi_class='auto', random_state=0, solver='liblinear'), RandomForestClassifier(criterion='entropy', random_state=0), SVC(gamma='auto', random_state=0)])

In [ ]:
# then, retrieving the value for a given key:
models['LogisticRegression']

LogisticRegression(multi_class='auto', random_state=0, solver='liblinear')

Finally, we can use `GridSearchCV`, that will fit every model with all possible combinations of parameters to check which one gives better results. This of course can be computing intensive task.




In [ ]:
from sklearn.model_selection import GridSearchCV

scores = []

for i in models.keys():
    clf =  GridSearchCV(models[i], params[i], cv=5, return_train_score=False)
    clf.fit(X_train_sc, y_train)
    scores.append({
        'model': models[i],
        'best_score': clf.best_score_,
        'best_params': clf.best_params_
    })

# return_train_score=False:
# - Especifica que no se almacenarán ni devolverán los puntajes (scores) del conjunto de entrenamiento en los resultados de la búsqueda.

# - ¿Qué significa?
#     Esto ayuda a ahorrar memoria y tiempo, ya que solo se almacenarán los puntajes de validación (los que se obtienen en cada pliegue de la validación cruzada) y no los del entrenamiento.
# - ¿Por qué podría ser útil?
#     Si solo te interesa evaluar la capacidad de generalización del modelo y no necesitas analizar cómo se comportó durante el entrenamiento, puedes establecerlo en False.

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

In [ ]:
df = pd.DataFrame(scores,columns=['model','best_score','best_params'])
df

,model,best_score,best_params
0,"LogisticRegression(multi_class='auto', random_...",0.826667,{'C': 1}
1,"RandomForestClassifier(criterion='entropy', ra...",0.893333,{'n_estimators': 10}
2,"SVC(gamma='auto', random_state=0)",0.903333,"{'C': 1, 'kernel': 'rbf'}"


To reduce processing time, we could instead use `RandomizedSearchCV` that fits just a subset of random combinations of parameters.

**Combinación aleatoria de hyperparametros vs combinación exahustiva**

En el caso anterior se va a realizar una busqueda exahustiva, combinando todos los hyperparametros con todos. Imaginad el caso de un dataset con 1.000.000 datos, donde queremos evaluar 100 hyperparámetros con 5 modelos diferentes, en este caso el GridSearch podria tardar una semana.

Solucion: RandomizedSearchCV, donde haremos una selección aleatoria de combinación de hyperparámetros.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

scores = []

for i in models.keys():
    clf =  RandomizedSearchCV(models[i], params[i], cv=5, return_train_score=False, n_iter=3)
    clf.fit(X_train_sc, y_train)
    scores.append({
        'model': models[i],
        'best_score': clf.best_score_,
        'best_params': clf.best_params_
    })

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

In [ ]:
df = pd.DataFrame(scores,columns=['model','best_score','best_params'])
df

,model,best_score,best_params
0,"LogisticRegression(multi_class='auto', random_...",0.826667,{'C': 1}
1,"RandomForestClassifier(criterion='entropy', ra...",0.893333,{'n_estimators': 10}
2,"SVC(gamma='auto', random_state=0)",0.903333,"{'kernel': 'rbf', 'C': 20}"
